# Human-in-the-Loop Checkpoints — Pausing an Agent for Approval

The SDK's tool-use loop supports optional human-in-the-loop checkpoints via `ClaudeAgentOptions.can_use_tool` — a callback that runs before any tool call not already pre-approved, letting you pause and require explicit sign-off before a sensitive action proceeds.


In [5]:
from typing import Any

from claude_agent_sdk import (
    tool,  # decorator that turns a Python function into a Claude-usable tool
    create_sdk_mcp_server,  # bundles one or more tools into a "server" Claude can talk to
    ClaudeSDKClient,  # a client you can keep open and send several messages through
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    ToolPermissionContext,  # extra info passed to our approval function
    PermissionResultAllow,  # return this to let a tool call proceed
    PermissionResultDeny,  # return this to block a tool call
    AssistantMessage,  # message type that holds Claude's actual reply
    ToolUseBlock,  # message piece that shows "Claude is calling a tool now"
    ResultMessage,  # the last message in the stream — carries the final answer plus stats (cost, duration, etc.)
)


def delete_file(path: str) -> bool:
    """Mock deletion — never touches a real file."""
    print(f"[mock] would delete: {path}")
    return True


# A deliberately "dangerous-sounding" tool — deleting a file — so we have
# something worth pausing for approval before it runs.
@tool("delete_file", "Delete a file at the given path", {"path": str})
async def delete_file_tool(args: dict[str, Any]) -> dict[str, Any]:
    deleted = delete_file(args["path"])
    return {"content": [{"type": "text", "text": f"Deleted: {deleted}"}]}


files_server = create_sdk_mcp_server(name="files", version="1.0.0", tools=[delete_file_tool])

## The approval checkpoint


In [6]:
# This function is our "approval checkpoint" — the SDK calls it automatically
# right before running any tool call that isn't already pre-approved. It's
# our chance to say yes/no before something sensitive actually happens.
async def approval_checkpoint(
    tool_name: str,  # which tool Claude wants to call
    input_data: dict[str, Any],  # the arguments it wants to call it with
    context: ToolPermissionContext,  # extra session context (not used here)
) -> PermissionResultAllow | PermissionResultDeny:
    if tool_name == "mcp__files__delete_file":
        # Real human-in-the-loop: this blocks and waits for actual keyboard
        # input instead of auto-approving. Type "y" to allow, anything else
        # to deny. In a real app you'd await something equivalent — a Slack
        # button click, a web UI response — instead of a terminal input().
        answer = input(f"Approval required: delete_file({input_data}) — approve? (y/n): ")
        if answer.strip().lower() != "y":
            return PermissionResultDeny(message="User denied the delete_file request.")

    # Returning PermissionResultAllow lets the tool call go ahead.
    # Returning PermissionResultDeny(...) instead would block it.
    return PermissionResultAllow(updated_input=input_data)


options = ClaudeAgentOptions(
    model="haiku",
    mcp_servers={"files": files_server},
    # can_use_tool: registers our checkpoint function to run before every
    # tool call that isn't already in allowed_tools.
    can_use_tool=approval_checkpoint,
)

## Run it — watch the pause happen before the mock deletion


In [7]:
async def run_delete_demo() -> None:
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Delete the file /tmp/old_report.csv")
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, ToolUseBlock):
                        print(f"[tool call] {block.name}({block.input})")
            elif isinstance(message, ResultMessage):
                print(f"\nResult: {message.result}")


await run_delete_demo()

[tool call] ToolSearch({'query': 'select:mcp__files__delete_file', 'max_results': 1})
[tool call] mcp__files__delete_file({'path': '/tmp/old_report.csv'})
[mock] would delete: /tmp/old_report.csv

Result: Done! The file `/tmp/old_report.csv` has been successfully deleted.


## Summary

- `can_use_tool` fires before any tool call that isn't already pre-approved — the checkpoint for sensitive actions.
- Here it's a simulated auto-approval; in production this is where you'd block on a real human decision (Slack message, web UI, etc.) before returning `PermissionResultAllow` or `PermissionResultDeny`.
